In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
import joblib


In [2]:
df = pd.read_csv("ecommerce_customer_behavior_dataset_v2.csv")
df["Date"] = pd.to_datetime(df["Date"])
df


,Order_ID,Customer_ID,Date,Age,Gender,City,Product_Category,Unit_Price,Quantity,Discount_Amount,Total_Amount,Payment_Method,Device_Type,Session_Duration_Minutes,Pages_Viewed,Is_Returning_Customer,Delivery_Time_Days,Customer_Rating
0,ORD_000001-1,CUST_00001,2023-05-29,40,Male,Ankara,Books,29.18,1,0.00,29.18,Digital Wallet,Mobile,14,9,True,13,4
1,ORD_000001-2,CUST_00001,2023-10-12,40,Male,Ankara,Home & Garden,644.40,1,138.05,506.35,Credit Card,Desktop,14,8,True,6,2
2,ORD_000001-3,CUST_00001,2023-12-05,40,Male,Ankara,Sports,332.82,5,0.00,1664.10,Credit Card,Mobile,15,10,True,9,4
3,ORD_000002-1,CUST_00002,2023-05-11,33,Male,Istanbul,Food,69.30,5,71.05,275.45,Digital Wallet,Desktop,16,13,True,4,4
4,ORD_000002-2,CUST_00002,2023-06-16,33,Male,Istanbul,Beauty,178.15,3,0.00,534.45,Credit Card,Mobile,14,7,True,6,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17044,ORD_004999-1,CUST_04999,2024-01-16,44,Male,Antalya,Beauty,68.65,2,0.00,137.30,Digital Wallet,Mobile,14,10,False,6,1
17045,ORD_005000-1,CUST_05000,2023-02-22,24,Female,Eskisehir,Beauty,156.12,1,0.00,156.12,Credit Card,Mobile,15,8,True,7,5
17046,ORD_005000-2,CUST_05000,2023-06-29,24,Female,Eskisehir,Home & Garden,1065.24,3,0.00,3195.72,Credit Card,Desktop,17,11,True,5,4
17047,ORD_005000-3,CUST_05000,2023-12-29,24,Female,Eskisehir,Books,47.73,2,8.80,86.66,Credit Card,Desktop,13,11,True,12,2


In [3]:
df.isnull().sum()


Order_ID                    0
Customer_ID                 0
Date                        0
Age                         0
Gender                      0
City                        0
Product_Category            0
Unit_Price                  0
Quantity                    0
Discount_Amount             0
Total_Amount                0
Payment_Method              0
Device_Type                 0
Session_Duration_Minutes    0
Pages_Viewed                0
Is_Returning_Customer       0
Delivery_Time_Days          0
Customer_Rating             0
dtype: int64

In [4]:
# This dataset has one row per ORDER, not per customer, and no explicit
# "Churn" label. "Is_Returning_Customer" turns out to just mark whether a
# given order is a customer's 2nd+ order (always True after their first
# purchase) - it's an order-sequence artifact, not a behavioral signal, so
# it has no predictive relationship with per-order features like price or
# session length (tested: ROC AUC ~0.48, i.e. random).
#
# Instead we build a proper customer-level churn label using RFM-style
# aggregation: a customer is "churned" if their most recent order is more
# than 90 days before the last date in the dataset (a common, defensible
# churn definition when no explicit label exists).
snapshot_date = df["Date"].max()
RECENCY_CHURN_THRESHOLD_DAYS = 90

customers = df.groupby("Customer_ID").agg(
    Age=("Age", "first"),
    Gender=("Gender", "first"),
    City=("City", "first"),
    num_orders=("Order_ID", "count"),
    total_spent=("Total_Amount", "sum"),
    avg_order_value=("Total_Amount", "mean"),
    avg_discount=("Discount_Amount", "mean"),
    avg_quantity=("Quantity", "mean"),
    avg_session_duration=("Session_Duration_Minutes", "mean"),
    avg_pages_viewed=("Pages_Viewed", "mean"),
    avg_delivery_time=("Delivery_Time_Days", "mean"),
    avg_rating=("Customer_Rating", "mean"),
    first_order=("Date", "min"),
    last_order=("Date", "max"),
    fav_category=("Product_Category", lambda x: x.mode()[0]),
    fav_payment=("Payment_Method", lambda x: x.mode()[0]),
    fav_device=("Device_Type", lambda x: x.mode()[0]),
).reset_index()

# Tenure = how long they've been an active customer (behavioral feature - OK to use)
customers["tenure_days"] = (customers["last_order"] - customers["first_order"]).dt.days

# Recency is only used to DEFINE the label, never as a model feature,
# otherwise it would leak the target (recency > 90 -> Churn by definition).
recency_days = (snapshot_date - customers["last_order"]).dt.days
customers["Churn"] = (recency_days > RECENCY_CHURN_THRESHOLD_DAYS).astype(int)

df1 = customers.drop(columns=["Customer_ID", "first_order", "last_order"]).copy()
df1["Churn"].value_counts(normalize=True)


Churn
1    0.5096
0    0.4904
Name: proportion, dtype: float64

In [5]:
splitt = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, test_index in splitt.split(df1, df1["Churn"]):
    train_set = df1.loc[train_index]
    test_set = df1.loc[test_index]

x_train = train_set.drop("Churn", axis=1)
y_train = train_set["Churn"]
x_test = test_set.drop("Churn", axis=1)
y_test = test_set["Churn"]

nums_attribute = x_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
cats_attribute = x_train.select_dtypes(include=["object"]).columns.tolist()

nums_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cats_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("nums", nums_pipeline, nums_attribute),
    ("cats", cats_pipeline, cats_attribute)
])

# RandomForest: handles mixed numeric/categorical features well, robust to
# outliers/scale, and class_weight="balanced" compensates for any imbalance.
full_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42
    ))
])

full_pipeline.fit(x_train, y_train)

y_pred = full_pipeline.predict(x_test)
y_proba = full_pipeline.predict_proba(x_test)[:, 1]

comparison = pd.DataFrame({"Actual": y_test, "Predicted": y_pred})
comparison.to_csv("comparison.csv", index=False)

print(y_test.value_counts(normalize=True))
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Churn
1    0.51
0    0.49
Name: proportion, dtype: float64
Accuracy: 0.754
ROC AUC: 0.8026430572228891

Confusion Matrix:
 [[317 173]
 [ 73 437]]

Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.65      0.72       490
           1       0.72      0.86      0.78       510

    accuracy                           0.75      1000
   macro avg       0.76      0.75      0.75      1000
weighted avg       0.76      0.75      0.75      1000



In [6]:
# Which features actually drive churn predictions
importances = full_pipeline.named_steps["model"].feature_importances_
feature_names = full_pipeline.named_steps["preprocessor"].get_feature_names_out()

feat_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

feat_importance.head(10)


,feature,importance
10,nums__tenure_days,0.255529
2,nums__total_spent,0.077377
3,nums__avg_order_value,0.065292
1,nums__num_orders,0.063106
4,nums__avg_discount,0.054428
6,nums__avg_session_duration,0.054248
8,nums__avg_delivery_time,0.053033
0,nums__Age,0.052319
7,nums__avg_pages_viewed,0.050973
5,nums__avg_quantity,0.046244


In [7]:
# Save the full pipeline (preprocessing + model) as a single .pkl file
# compress=3 keeps file size reasonable without hurting accuracy
joblib.dump(full_pipeline, "churn_model.pkl", compress=3)
print("Model saved to churn_model.pkl")


Model saved to churn_model.pkl
